In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze — Ingestion
# MAGIC Reads one month of raw NYC TLC Yellow Taxi trip data and appends it to Bronze, with ingestion lineage metadata.

# COMMAND ----------

from pyspark.sql import functions as F

VOLUME = "/Volumes/nyc_taxi/bronze/raw_files"

def ingest_month(year: int, month: int):
    """Read one month of raw trip data and append it to the Bronze table."""
    filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
    path = f"{VOLUME}/{filename}"

    df = spark.read.parquet(path)

    df = (df
        .withColumn("_source_file",  F.lit(filename))
        .withColumn("_ingested_at",  F.current_timestamp())
        .withColumn("_source_year",  F.lit(year))
        .withColumn("_source_month", F.lit(month))
    )

    (df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("nyc_taxi.bronze.yellow_tripdata"))

    print(f"{year}-{month:02d}: appended {df.count():,} rows")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Job parameters
# MAGIC Reads year/month from the Workflow task's parameters. Defaults to 2024/1 only as a fallback for manual testing.

# COMMAND ----------

dbutils.widgets.text("year", "2024")
dbutils.widgets.text("month", "1")

year  = int(dbutils.widgets.get("year"))
month = int(dbutils.widgets.get("month"))

print(f"Parameters read: year={year}, month={month}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Guard — refuse to re-ingest a month already in Bronze
# MAGIC Added 2026-07-28 after a standalone test run silently duplicated January 2024, cascading through Silver and Gold. This check stops that failure mode at its actual source, before any data gets written.

# COMMAND ----------

already_ingested = (spark.table("nyc_taxi.bronze.yellow_tripdata")
    .filter((F.col("_source_year") == year) & (F.col("_source_month") == month))
    .count() > 0)

if already_ingested:
    raise Exception(f"{year}-{month:02d} is already in Bronze. Refusing to re-ingest — check parameters before running.")

print(f"{year}-{month:02d} not yet in Bronze — safe to proceed.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Run the ingestion
# MAGIC Only reached if the guard above didn't raise an exception.

# COMMAND ----------

ingest_month(year, month)